# Task 4: Run agents with Amazon Bedrock AgentCore Harness

In Task 3 you deployed a multi-agent system to **AgentCore Runtime**: you wrote the orchestration loop in `main.py`, wrapped it with `BedrockAgentCoreApp`, and let AgentCore package and host the container.

**AgentCore Harness** solves the same problem from the other direction. The agent loop itself is managed for you (it is powered by Strands Agents, the same framework used earlier in this class). You *declare* what the agent is — model, system prompt, tools, memory, and execution limits — as **configuration**, and AgentCore runs the reasoning → tool-call → result → response loop. There is no orchestration code to write and no container to build. Switching a model or adding a tool is a configuration change, not a redeploy.

AgentCore Harness became generally available in June 2026, after the earlier tasks in this lab were written, so this notebook is a standalone addition that builds on the concepts from Task 3.

## What You Will Learn

- The conceptual difference between **AgentCore Harness** (managed loop, config-only) and **AgentCore Runtime** (your code, your loop) from Task 3.
- How to create the IAM **execution role** a harness assumes, with the confused-deputy trust conditions AWS recommends.
- The authoritative three-stage lifecycle: **create** (control plane) → **wait for `READY`** (poll) → **invoke** (data plane, streamed).
- How to read the **streaming event format** returned by `invoke_harness`.
- How to use **per-invocation overrides** to change the model or system prompt for a single call without redeploying — the feature that makes prototyping and A/B testing simple.
- How to tear the harness down so you are not billed for an idle resource.

## Harness vs. Runtime

Both run on the same underlying AgentCore Runtime infrastructure, but they divide the work differently:

| | **Harness** (this task) | **Runtime** (Task 3) |
|---|---|---|
| You provide | Configuration, no code | Agent code + ARM64 container |
| Agent loop | Managed (Strands) | You write it |
| Change model / add a tool | Config change, no redeploy | Code change + redeploy |
| Framework choice | Strands only | Any (LangGraph, CrewAI, custom) |
| Best for | Fastest path from idea to running agent, dynamic config | Full control of the loop, non-loop patterns (graph/workflow), bidirectional streaming |

Rule of thumb: **start with Harness; drop down to Runtime only when configuration is not enough.**

> ℹ️ The APIs are split across two clients, exactly as in Task 3:
> - **Control plane** `bedrock-agentcore-control` — `create_harness`, `get_harness`, `delete_harness` (manage the resource).
> - **Data plane** `bedrock-agentcore` — `invoke_harness` (talk to the running agent).

In [ ]:
%%capture
# Install dependencies (boto3 provides the AgentCore control- and data-plane clients)
!pip install --force-reinstall -U -r requirements.txt --quiet --disable-pip-version-check

In [ ]:
# Imports and AWS session setup
import json
import time
import uuid

import boto3
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

# Control plane: manage the harness resource. Data plane: talk to the running agent.
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)
agentcore_runtime = boto3.client("bedrock-agentcore", region_name=region)

print(f"Region:     {region}")
print(f"Account ID: {account_id}")

### Task 4.1: Create the execution role

A harness runs under an IAM **execution role** that the AgentCore service assumes on your behalf. Two things matter here:

1. **Trust policy** — the role must allow the service principal `bedrock-agentcore.amazonaws.com` to assume it. We scope it with **confused-deputy conditions** (`aws:SourceAccount` and `aws:SourceArn`) so that only *your own* harnesses in *this* account can assume the role. Without these conditions, any harness in any account could assume it.
2. **Permissions policy** — keep it least-privilege. Here we grant only Bedrock model invocation. In production, restrict the model `Resource` to specific inference-profile ARNs rather than `*`.

> This mirrors the authentication work you did with Cognito in Task 3 — the difference is that there we authenticated *callers*, and here we are authorizing the *agent runtime* itself.

> **No permission to create IAM roles?** Restricted lab or sandbox accounts often deny `iam:CreateRole`. In that case, set `EXISTING_ROLE_ARN` at the top of the code cell below to a role you already have — one trusted by `bedrock-agentcore.amazonaws.com` with `bedrock:InvokeModel` permissions — and the notebook will reuse it and skip role creation.

In [ ]:
# --- Execution role ---------------------------------------------------------
# A harness needs an IAM execution role. If your identity cannot create IAM
# roles (common in restricted lab/sandbox accounts), set EXISTING_ROLE_ARN to a
# role you already have and this cell will reuse it instead of creating one.
#
#   EXISTING_ROLE_ARN = "arn:aws:iam::<account>:role/<YourHarnessRole>"
#
# The role's trust policy must allow bedrock-agentcore.amazonaws.com to assume it
# (ideally scoped with the confused-deputy conditions shown below), and it needs
# bedrock:InvokeModel permissions.
EXISTING_ROLE_ARN = None  # <-- set to a role ARN to skip role creation

iam = boto3.client("iam")
role_name = "AgentCoreHarnessLabRole"
created_role = False  # track whether WE created it, so cleanup only deletes our own

# Confused-deputy-safe trust policy: only this account's harnesses may assume the role.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {
                    "aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:*"
                },
            },
        }
    ],
}

# Least-privilege permissions: Bedrock model invocation only.
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream",
            ],
            "Resource": "*",
        }
    ],
}

if EXISTING_ROLE_ARN:
    execution_role_arn = EXISTING_ROLE_ARN
    print(f"Using existing execution role: {execution_role_arn}")
else:
    try:
        try:
            role = iam.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description="Execution role for the AgentCore Harness lab",
            )
            execution_role_arn = role["Role"]["Arn"]
            created_role = True
            print(f"Created role: {execution_role_arn}")
        except iam.exceptions.EntityAlreadyExistsException:
            execution_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
            created_role = True
            print(f"Role already exists, reusing: {execution_role_arn}")

        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="HarnessBedrockInvoke",
            PolicyDocument=json.dumps(permissions_policy),
        )
        # IAM is eventually consistent; let the role propagate before the harness assumes it.
        time.sleep(10)
        print("Attached permissions policy.")
    except ClientError as e:
        if e.response["Error"]["Code"] in ("AccessDenied", "AccessDeniedException"):
            raise PermissionError(
                "This identity cannot create IAM roles. Set EXISTING_ROLE_ARN at the "
                "top of this cell to a role you already have (trusted by "
                "bedrock-agentcore.amazonaws.com with bedrock:InvokeModel) and re-run."
            ) from e
        raise


### Task 4.2: Create the harness (control plane)

`create_harness` requires only a **name** and an **execution role ARN**. Everything else is optional, declarative configuration. We give our finance assistant a `systemPrompt` and set the three **execution limits** explicitly rather than relying on service defaults — `maxIterations`, `maxTokens`, and `timeoutSeconds` are your cost and abuse guardrails.

If no model is configured, the harness applies a default Bedrock model. We leave the model unset here so this notebook works regardless of which models are enabled in your account; Task 4.4 shows how to select one per invocation.

> **Naming constraint:** `harnessName` must start with a letter and contain only letters, digits, and underscores (max 40 characters).

> ℹ️ **API note.** The parameter and response shapes in this notebook were verified against `botocore` 1.43.x. If you are on a different version and a call raises a `ParamValidationError`, run `agentcore_control.meta.service_model.operation_model("CreateHarness").input_shape.members.keys()` to list the parameters your installed `boto3` accepts. Note that `create_harness`/`get_harness` nest the result under a `harness` key in the response.

In [ ]:
harness_name = "FinanceAssistantHarness"

system_prompt = (
    "You are a careful personal finance assistant. You explain budgeting, saving, "
    "and diversified investing concepts in plain language. You never give Bitcoin or "
    "other cryptocurrency investment advice. When a user asks for a recommendation, "
    "you remind them to consult a licensed financial advisor."
)

create_response = agentcore_control.create_harness(
    harnessName=harness_name,
    executionRoleArn=execution_role_arn,
    systemPrompt=[{"text": system_prompt}],
    maxIterations=10,
    maxTokens=2048,
    timeoutSeconds=300,
)

harness = create_response["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID:  {harness_id}")
print(f"Harness ARN: {harness_arn}")

### Task 4.3: Wait for the harness to become `READY`

Creation is asynchronous. The status progresses `CREATING → READY`, and **you cannot invoke a harness until it reports `READY`**. The terminal failure states are `CREATE_FAILED`, `UPDATE_FAILED`, and `DELETE_FAILED` — if you hit one, inspect the harness with `get_harness` for the failure reason.

Skipping this poll and invoking immediately is the single most common mistake with the Harness API.

In [ ]:
def wait_for_ready(harness_id, timeout_seconds=300, poll_seconds=10):
    """Poll get_harness until the harness is READY or a terminal state / timeout is reached."""
    deadline = time.time() + timeout_seconds
    terminal_failures = {"CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"}
    while time.time() < deadline:
        harness = agentcore_control.get_harness(harnessId=harness_id)["harness"]
        status = harness["status"]
        print(f"  status: {status}")
        if status == "READY":
            return status
        if status in terminal_failures:
            reason = harness.get("failureReason", "(no reason provided)")
            raise RuntimeError(f"Harness entered terminal state: {status} — {reason}")
        time.sleep(poll_seconds)
    raise TimeoutError("Harness did not become READY in time")


print("Waiting for harness to become READY...")
wait_for_ready(harness_id)
print("\nHarness is READY — it can now be invoked.")

### Task 4.4: Invoke the harness (data plane, streamed)

Invocation happens on the **data plane** client (`bedrock-agentcore`), not the control plane. Two things to know:

- **`runtimeSessionId` must be at least 33 characters.** A standard `uuid4()` is 36 characters including hyphens, which satisfies this. **Reuse the same session id** across calls to continue the same conversation in the same isolated environment.
- **The response is a stream of events, not a single string.** You must iterate `response["stream"]`. The event types are:
  - `messageStart` — a new message begins (includes the role)
  - `contentBlockDelta` — an incremental chunk (text, tool use, tool result, or reasoning)
  - `contentBlockStop` / `messageStop` — end of a block / message (`messageStop` carries a `stopReason` such as `end_turn`, `tool_use`, `max_iterations_exceeded`, or `timeout_exceeded`)
  - `metadata` — token usage and latency

The helper below iterates the stream and prints text deltas as they arrive, so you see the response appear incrementally — the same streaming experience as the playground.

In [ ]:
def invoke_and_stream(harness_arn, prompt, session_id, overrides=None):
    """Invoke the harness and print streamed text deltas. Returns the full text."""
    kwargs = {
        "harnessArn": harness_arn,
        "runtimeSessionId": session_id,
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
    }
    if overrides:
        kwargs.update(overrides)

    response = agentcore_runtime.invoke_harness(**kwargs)

    collected = []
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            text = delta.get("text")
            if text:
                print(text, end="", flush=True)
                collected.append(text)
        elif "messageStop" in event:
            reason = event["messageStop"].get("stopReason")
            print(f"\n[stopReason: {reason}]")
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})
            if usage:
                print(f"[tokens: {usage}]")
    return "".join(collected)


# Reuse this session id across calls to keep one continuous conversation.
session_id = str(uuid.uuid4())  # 36 chars — satisfies the >= 33 requirement
print(f"Session: {session_id}\n")

_ = invoke_and_stream(
    harness_arn,
    "I make $6000/month and want to start investing $500/month. "
    "Help me think about a simple, diversified starter approach.",
    session_id,
)

### Task 4.5: Per-invocation overrides

This is the feature that makes a harness so convenient for experimentation. `invoke_harness` can **override the harness configuration for a single call** without changing the stored resource — no redeploy. Overridable fields include `model`, `systemPrompt`, `tools`, `allowedTools`, `skills`, and the execution limits.

Below we override the `systemPrompt` for one call to make the assistant answer as a concise bullet-point summarizer, while the harness's stored configuration stays exactly as created. Change the next call back and the original behavior returns.

> **Security note:** because overrides are caller-supplied, they are a **trust boundary**. If you ever expose a harness to untrusted users, strip or allowlist override fields (especially `model` and `skills`) in your application layer before calling `invoke_harness`.

In [ ]:
override_prompt = (
    "You are a terse assistant. Answer only in at most 4 short bullet points. "
    "No preamble, no closing remarks."
)

# A fresh session id so this experiment does not inherit the earlier conversation.
experiment_session = str(uuid.uuid4())

_ = invoke_and_stream(
    harness_arn,
    "What is dollar-cost averaging?",
    experiment_session,
    overrides={"systemPrompt": [{"text": override_prompt}]},
)

#### What Just Happened?

Compared with Task 3's Runtime deployment, notice what you did **not** do here:

- You wrote **no `main.py`** and **no agent loop** — the Strands-powered loop is managed.
- You built **no container** and pushed **nothing to ECR**.
- Changing the agent's behavior in Task 4.5 was a **single API parameter**, not a code change and redeploy.

That is the core trade-off: Harness trades the flexibility of writing your own loop (Task 3) for the speed of pure configuration. For a straightforward conversational finance assistant, configuration is enough.

### Task 4.6: Clean up

A harness is a billable resource, so delete it when you are done experimenting. We delete the harness first, then the IAM role we created for it.

> See `aws_cleanup.ipynb` for cleanup of the other resources created across this lab (Cognito pools, guardrails, Runtime agents).

In [ ]:
# Delete the harness
try:
    agentcore_control.delete_harness(harnessId=harness_id)
    print(f"Deleted harness: {harness_id}")
except ClientError as e:
    print(f"Could not delete harness (it may already be gone): {e}")

# Delete the execution role ONLY if this notebook created it. If you supplied
# EXISTING_ROLE_ARN, we leave your role untouched.
if created_role:
    try:
        iam.delete_role_policy(RoleName=role_name, PolicyName="HarnessBedrockInvoke")
        iam.delete_role(RoleName=role_name)
        print(f"Deleted role: {role_name}")
    except ClientError as e:
        print(f"Could not delete role (it may already be gone): {e}")
else:
    print("Left the pre-existing execution role in place (not created by this notebook).")


**Task complete:** You created an AgentCore Harness from pure configuration, waited for it to become `READY`, invoked it with a streamed response, changed its behavior with a per-invocation override, and cleaned it up — all without writing an agent loop or building a container. You now have both tools in your kit: **Runtime** (Task 3) when you need to own the loop, and **Harness** (Task 4) when configuration is enough.

### Next Steps

From here you can extend the harness with:

- **Tools** — attach the built-in `shell` and `file_operations` tools, or opt in to AgentCore Browser, Code Interpreter, Gateway, or remote MCP servers.
- **Memory** — enable short-term and long-term AgentCore Memory, scoped per user via `actorId`, for conversations that persist across sessions.
- **Skills** — attach Agent Skills (the open AgentSkills.io standard) from AWS Skills, a Git repo, Amazon S3, or the session filesystem.
- **Versions and endpoints** — each update produces an immutable version; point named endpoints at a version and roll back instantly by repointing.

See the [AgentCore Harness documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/harness.html) for the authoritative, current API reference.